In [0]:
from pyspark.sql.types import StructType,StructField,StringType,IntegerType,DataType,TimestampType,FloatType
import pyspark.sql.functions as F

In [0]:
catalog_name="ecommerce"

df_bronze=spark.table(f"{catalog_name}.bronze.brz_brands")
df_bronze.show(10)

In [0]:
df_silver=df_bronze.withColumn("brand_name",F.trim(F.col("brand_name")))
df_silver.show(10)

In [0]:
df_silver=df_silver.withColumn("brand_code",F.regexp_replace(F.col("brand_code"),r"[^A-Za-z0-9]"," "))
df_silver.display(5)

In [0]:
df_silver.select("category_code").distinct().show()

In [0]:
#anomalies
anomalies={
    "GROCERY":"GRCY",
    "BOOKS":"BKS",
    "TOYS":"TOY"
}
# PySpark replace is easy
df_silver = df_silver.replace(to_replace=anomalies, subset=["category_code"])

# ✅ Show results
df_silver.select("category_code").distinct().show()

In [0]:
df_silver.write.format("delta").mode("overwrite").option("mergeSchema","True").saveAsTable(f"{catalog_name}.silver.slv_brands")